In [1]:
import torch
from diffusers import DiffusionPipeline, DPMSolverMultistepScheduler
from transformers import CLIPTokenizer, CLIPTextModel
import numpy as np


device = "cuda:2" if torch.cuda.is_available() else "cpu"
model_id = "openai/clip-vit-large-patch14"
tokenizer = CLIPTokenizer.from_pretrained(model_id)
text_encoder = CLIPTextModel.from_pretrained(model_id).to(device)

# Function to get text embedding
def get_text_embedding(texts, tokenizer=tokenizer, text_encoder=text_encoder, device=device):
    
    if isinstance(texts, str):
        texts = [texts]

    inputs = tokenizer(
        texts,
        padding="max_length",
        max_length=tokenizer.model_max_length,
        truncation=True,
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():
        outputs = text_encoder(**inputs)
        # You might get different outputs: last_hidden_state, or pooled output
        # For SD v1.5, the documentation says “non-pooled output of the text encoder is fed into the UNet” :contentReference[oaicite:3]{index=3}
        embedding = outputs.last_hidden_state  # shape: (batch_size, seq_len, hidden_dim)
    return embedding


# Load the text-to-video model
model_id = "cerspense/zeroscope_v2_576w"  # Text-to-video model
pipe = DiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
)
pipe.to("cuda:2")

# Optional: Use faster scheduler
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)

# Your original 768-dim embedding
# your_768_embedding = torch.randn(1, 77, 768, dtype=torch.float16, device="cuda:2")

your_768_embedding = get_text_embedding("""A boat sailing in the ocean""", tokenizer=tokenizer, text_encoder=text_encoder, device=device)


# Convert 768-dim to 1024-dim by padding with zeros
padding = torch.zeros(1, 77, 256, dtype=torch.float16, device="cuda:2")
custom_embedding = torch.cat([your_768_embedding, padding], dim=-1)
print(f"Custom embedding shape: {custom_embedding.shape}")

# For classifier-free guidance, also convert negative embedding
negative_768_embedding = torch.randn(1, 77, 768, dtype=torch.float16, device="cuda:2")
negative_embedding = torch.cat([negative_768_embedding, padding], dim=-1)

# Generate video using custom embeddings
video_frames = pipe(
    prompt_embeds=custom_embedding,
    negative_prompt_embeds=negative_embedding,
    num_inference_steps=25,
    num_frames=24,
    height=320,
    width=576,
    guidance_scale=9.0,
).frames[0]

# Save video
from diffusers.utils import export_to_video
export_to_video(video_frames, "output_video.mp4", fps=8)

print("Video generated successfully!")
print(f"Video shape: {len(video_frames)} frames")

/home/subhankar/miniconda3/envs/ie643/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]An error occurred while trying to fetch /home/subhankar/.cache/huggingface/hub/models--cerspense--zeroscope_v2_576w/snapshots/6963642a64dbefa93663d1ecebb4ceda2d9ecb28/vae: Error no file named diffusion_pytorch_model.safetensors found in directory /home/subhankar/.cache/huggingface/hub/models--cerspense--zeroscope_v2_576w/snapshots/6963642a64dbefa93663d1ecebb4ceda2d9ecb28/vae.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]An error occurred while trying to fetch /home/subhankar/.cache/huggingface/hub/models--cerspense--zeroscope_v2_576w/snapsh

CLIPTextConfig {
  "architectures": [
    "CLIPTextModel"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 0,
  "dropout": 0.0,
  "dtype": "float16",
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_size": 1024,
  "initializer_factor": 1.0,
  "initializer_range": 0.02,
  "intermediate_size": 4096,
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 77,
  "model_type": "clip_text_model",
  "num_attention_heads": 16,
  "num_hidden_layers": 23,
  "pad_token_id": 1,
  "projection_dim": 512,
  "transformers_version": "4.57.1",
  "vocab_size": 49408
}

Custom embedding shape: torch.Size([1, 77, 1024])
torch.Size([1, 77, 1024]) torch.Size([1, 77, 1024])
Custom embedding shape: torch.Size([1, 77, 1024])
torch.Size([1, 77, 1024]) torch.Size([1, 77, 1024])


100%|██████████| 25/25 [00:33<00:00,  1.32s/it]



Video generated successfully!
Video shape: 24 frames


In [ ]:
# If you have 768-dimensional embeddings, you need to project them to 1024 dimensions
# Here are two approaches:

# Approach 1: Linear projection (learnable)
projection_layer = torch.nn.Linear(768, 1024, dtype=torch.float16, device="cuda:2")

# Your original 768-dim embedding
your_768_embedding = torch.randn(1, 77, 768, dtype=torch.float16, device="cuda:2")

# Project to 1024 dimensions
custom_embedding_1024 = projection_layer(your_768_embedding)
print(f"Projected embedding shape: {custom_embedding_1024.shape}")

# Approach 2: Padding with zeros (simpler, no learning)
# Pad from 768 to 1024 by adding 256 zeros
padding = torch.zeros(1, 77, 256, dtype=torch.float16, device="cuda:2")
custom_embedding_1024 = torch.cat([your_768_embedding, padding], dim=-1)
print(f"Padded embedding shape: {custom_embedding_1024.shape}")

### Using Newly trained mapper model

In [7]:
import os
import argparse
import torch
import torch.nn as nn
from pathlib import Path
from typing import List
import argparse
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.utils.data import DataLoader
from torch.utils.data.distributed import DistributedSampler
from torchvision import transforms
from PIL import Image
from diffusers import StableDiffusionPipeline
import wandb
from tqdm import tqdm
from mapper_model import ImageToTextMapper
from laion_dataset import ImageCaptionDataset
from typing import List, Tuple, Optional
from torch.utils.tensorboard import SummaryWriter

In [ ]:
import os
import tempfile
from pathlib import Path
from typing import List

import gradio as gr
import torch
from PIL import Image

from run_mapper import generate_variation as run_generate_variation
# import internal helpers to warm/load checkpoint weights
from run_mapper import _load_mapper, _restore_clip_text_from_ckpt, _prepare_image, clip_get_image_features
from transformers import CLIPModel, CLIPTextModel, CLIPProcessor, CLIPTokenizer

/home/subhankar/miniconda3/envs/ie643/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from mapper_model import ImageToTextMapper

In [3]:
def is_main_process() -> bool:
    """Return True if distributed is not in use, or this is rank 0."""
    return (not dist.is_initialized()) or dist.get_rank() == 0


def init_models(clip_model_name: str, device: torch.device):
    print(f"[Rank {dist.get_rank() if dist.is_initialized() else 0}] Loading CLIP model: {clip_model_name}")
    processor = CLIPProcessor.from_pretrained(clip_model_name)
    clip_model = CLIPModel.from_pretrained(clip_model_name).to(device)
    tokenizer = CLIPTokenizer.from_pretrained(clip_model_name)
    text_model = CLIPTextModel.from_pretrained(clip_model_name).to(device)

    clip_model.eval()
    text_model.eval()
    for p in clip_model.parameters():
        p.requires_grad = False
    for p in text_model.parameters():
        p.requires_grad = False

    # Dummy forward to infer dims
    dummy_img = Image.new("RGB", (224, 224), color="white")
    inputs = processor(images=dummy_img, return_tensors="pt")
    with torch.no_grad():
        img_inputs = {k: v.to(device) for k, v in inputs.items()}
        img_feats = clip_get_image_features(clip_model, **img_inputs)

    in_dim = img_feats.shape[-1]
    out_seq_len = tokenizer.model_max_length
    out_dim = text_model.config.hidden_size
    if is_main_process():
        print(f"in_dim = {in_dim}, out_seq_len = {out_seq_len}, out_dim = {out_dim}")
    return processor, clip_model, tokenizer, text_model, in_dim, out_seq_len, out_dim

In [4]:
CLIP_NAME = "openai/clip-vit-large-patch14"
SD_NAME = "runwayml/stable-diffusion-v1-5"
MAPPER_PATH = Path("/home/subhankar/koustav/Image_to_Image_Diffusion/mapper_model_1024/mapper_epoch4.pth")
OUT_DIR = Path("/home/subhankar/koustav/Image_to_Image_Diffusion/gradio_out")
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [8]:
def generate_variation(
    mapper_path: str,
    clip_model_name: str,
    sd_model_name: str,
    input_image_path: str,
    out_dir: str,
    device: torch.device,
    num_inference_steps: int = 50,
    guidance_scale: float = 3.0,
    strength: float = 0.7,
    seed: int = 42,
    use_source_latents: bool = False,  # keep default for backward compat
    variations: int = 3,               # <-- added parameter
    use_checkpoint_clip: bool = False,
) -> List[str]:
    """
    Generate image variation(s) using the Stable Diffusion pipeline.
    The mapper is used only to produce prompt_embeds conditioning for the pipeline.
    Returns list of saved file paths.
    """
    os.makedirs(out_dir, exist_ok=True)
    device = torch.device(device)

    # choose pipeline dtype (float16 on CUDA for speed)
    pipe_dtype = torch.float16 if device.type == "cuda" else torch.float32

    # load CLIP and text models (kept in float32 for stability)
    processor = CLIPProcessor.from_pretrained(clip_model_name)
    clip_model = CLIPModel.from_pretrained(clip_model_name).to(device).eval()
    tokenizer = CLIPTokenizer.from_pretrained(clip_model_name)
    text_model = CLIPTextModel.from_pretrained(clip_model_name).to(device).eval()
    for p in list(clip_model.parameters()) + list(text_model.parameters()):
        p.requires_grad = False

    # Optionally restore fine-tuned CLIP/text weights from the checkpoint if provided
    if use_checkpoint_clip:
        try:
            restored = _restore_clip_text_from_ckpt(mapper_path, clip_model, text_model, device)
            if restored and is_main_process():
                print("Using CLIP/Text weights restored from checkpoint for generation.")
        except Exception as e:
            if is_main_process():
                print(f"Warning: could not restore CLIP/text from checkpoint: {e}")

    # load mapper checkpoint
    if not os.path.exists(mapper_path):
        raise FileNotFoundError(f"Mapper checkpoint not found: {mapper_path}")
    mapper, cfg = _load_mapper(mapper_path, device=torch.device("cpu"))
    mapper.to(device).eval()

    # load pipeline
    pipe = StableDiffusionPipeline.from_pretrained(sd_model_name, torch_dtype=pipe_dtype).to(device)
    pipe.safety_checker = None
    vae, unet, scheduler = pipe.vae, pipe.unet, pipe.scheduler

    # prepare image tensors
    pil_img, img_tensor = _prepare_image(input_image_path)
    img_tensor = img_tensor.to(device=device, dtype=torch.float32)  # for CLIP/mapper

    # compute mapped conditioning and uncond embedding
    with torch.no_grad():
        # Use same preprocessing as training (resize/center-crop/normalize)
        # across transformers versions: some image processors don't accept extra kwargs
        try:
            clip_inputs = processor(images=[pil_img], return_tensors="pt", do_normalize=True, do_resize=True, do_center_crop=True)
        except Exception:
            try:
                clip_inputs = processor(images=[pil_img], return_tensors="pt")
            except Exception:
                # Fall back to calling image_processor / feature_extractor directly
                img_proc = getattr(processor, "image_processor", None) or getattr(processor, "feature_extractor", None)
                if img_proc is None:
                    raise
                clip_inputs = img_proc(images=[pil_img], return_tensors="pt")
        clip_inputs = {k: v.to(device) for k, v in clip_inputs.items()}
        # safe call in case clip_model was wrapped in DDP
        img_feats = clip_get_image_features(clip_model, **clip_inputs)  # float32
        mapped = mapper(img_feats)  # [1, L, H] float32

        uncond_tokens = tokenizer([""], padding="max_length", truncation=True,
                                  max_length=tokenizer.model_max_length, return_tensors="pt")
        uncond_tokens = {k: v.to(device) for k, v in uncond_tokens.items()}
        uncond_emb = text_model(**uncond_tokens).last_hidden_state  # [1, L, H] float32

    # align dtype/device for pipeline UNet
    target_dtype = unet.dtype
    # Ensure contiguous, correct device and dtype expected by the pipeline
    mapped = mapped.contiguous().to(device=device, dtype=target_dtype)
    uncond_emb = uncond_emb.contiguous().to(device=device, dtype=target_dtype)

    return mapped

    # prepare latents (either encoded source or random base — we'll sample per-variation)
    with torch.no_grad():
        enc_in = img_tensor.to(device=device, dtype=pipe_dtype)
        latents_orig = vae.encode(enc_in).latent_dist.sample() * vae.config.scaling_factor  # [1, C, H/8, W/8]

    out_paths: List[str] = []
    for i in range(variations):
        gen_seed = int(seed) + i
        if is_main_process():
            print(f"Generating variation {i} with seed {gen_seed}")
        generator = torch.Generator(device=device).manual_seed(gen_seed)

        if use_source_latents:
            latents = latents_orig.clone().to(device=device, dtype=latents_orig.dtype)
        else:
            latents = torch.randn_like(latents_orig, device=device, dtype=latents_orig.dtype)

        # Run pipeline (it will perform denoising + decode)
        try:
            images = pipe(
                prompt_embeds=mapped,
                negative_prompt_embeds=uncond_emb,
                num_inference_steps=num_inference_steps,
                guidance_scale=guidance_scale,
                latents=latents,
                generator=generator,
                output_type="pil",
            ).images
        except Exception as e:
            # Surface an informative error for debugging
            raise RuntimeError(f"Stable Diffusion pipeline failed for variation {i} (seed={gen_seed}): {e}")

        img_out = images[0]
        out_name = f"variation_{i}_seed{gen_seed}.png"
        out_path = os.path.join(out_dir, out_name)
        img_out.save(out_path)
        out_paths.append(out_path)
        if is_main_process():
            print(f"Saved generated variation: {out_path}")

    return out_paths

In [9]:
mapped = generate_variation(
    mapper_path=str(MAPPER_PATH),
    clip_model_name=CLIP_NAME,
    sd_model_name=SD_NAME,
    input_image_path="/home/subhankar/koustav/Image_to_Video_Diffusion/first_frame.png",
    out_dir=str(OUT_DIR),
    device=torch.device("cuda:3"),
    num_inference_steps=50,
    guidance_scale=3.0,
    strength=0.7,
    seed=42,
    use_source_latents=False,  # keep default for backward compat
    variations=3,               # <-- added parameter
    use_checkpoint_clip=False,
)

Loading pipeline components...: 100%|██████████| 7/7 [00:00<00:00, 11.50it/s]


In [10]:
mapped.shape

torch.Size([1, 77, 1024])

In [11]:
import torch
from diffusers import DiffusionPipeline, DPMSolverMultistepScheduler
from transformers import CLIPTokenizer, CLIPTextModel
import numpy as np
from diffusers.utils import export_to_video


device = "cuda:2" if torch.cuda.is_available() else "cpu"

# Load the text-to-video model
model_id = "cerspense/zeroscope_v2_576w"  # Text-to-video model
pipe = DiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
)
pipe.to("cuda:2")

# Optional: Use faster scheduler
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)



# Your original 768-dim embedding
# your_768_embedding = torch.randn(1, 77, 768, dtype=torch.float16, device="cuda:2")

# your_768_embedding = get_text_embedding("""A boat sailing in the ocean""", tokenizer=tokenizer, text_encoder=text_encoder, device=device)


# Convert 768-dim to 1024-dim by padding with zeros
# padding = torch.zeros(1, 77, 256, dtype=torch.float16, device="cuda:2")
# custom_embedding = torch.cat([your_768_embedding, padding], dim=-1)
print(f"Custom embedding shape: {custom_embedding.shape}")

# For classifier-free guidance, also convert negative embedding
negative_embedding = torch.randn(1, 77, 1024, dtype=torch.float16, device="cuda:2")
# negative_embedding = torch.cat([negative_1024_embedding], dim=-1)

# Generate video using custom embeddings
video_frames = pipe(
    prompt_embeds=mapped,
    negative_prompt_embeds=negative_embedding,
    num_inference_steps=25,
    num_frames=24,
    height=320,
    width=576,
    guidance_scale=9.0,
).frames[0]

# Save video
from diffusers.utils import export_to_video
export_to_video(video_frames, "output_video.mp4", fps=8)

print("Video generated successfully!")
print(f"Video shape: {len(video_frames)} frames")

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]An error occurred while trying to fetch /home/subhankar/.cache/huggingface/hub/models--cerspense--zeroscope_v2_576w/snapshots/6963642a64dbefa93663d1ecebb4ceda2d9ecb28/vae: Error no file named diffusion_pytorch_model.safetensors found in directory /home/subhankar/.cache/huggingface/hub/models--cerspense--zeroscope_v2_576w/snapshots/6963642a64dbefa93663d1ecebb4ceda2d9ecb28/vae.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
Loading pipeline components...:  80%|████████  | 4/5 [00:00<00:00,  6.48it/s]An error occurred while trying to fetch /home/subhankar/.cache/huggingface/hub/models--cerspense--zeroscope_v2_576w/snapshots/6963642a64dbefa93663d1ecebb4ceda2d9ecb28/unet: Error no file named diffusion_pytorch_model.safetensors found in directory /home/subhankar/.cache/huggingface/hub/models--cerspense--zeroscope_v2_576w/snapshots/6963642a64dbefa93663d1ecebb4ceda2d9ecb28/unet.
Defau

CLIPTextConfig {
  "architectures": [
    "CLIPTextModel"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 0,
  "dropout": 0.0,
  "dtype": "float16",
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_size": 1024,
  "initializer_factor": 1.0,
  "initializer_range": 0.02,
  "intermediate_size": 4096,
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 77,
  "model_type": "clip_text_model",
  "num_attention_heads": 16,
  "num_hidden_layers": 23,
  "pad_token_id": 1,
  "projection_dim": 512,
  "transformers_version": "4.57.1",
  "vocab_size": 49408
}



AcceleratorError: CUDA error: CUDA-capable device(s) is/are busy or unavailable
Search for `cudaErrorDevicesUnavailable' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
